<a href="https://colab.research.google.com/github/D-L2001/olist-operations-dashboard/blob/main/olist_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Late Deliveries, Low Stars: E-commerce Operations Analysis (Team 2, Pandas Pioneers)

## 1. Business question
- **Client:** the COO of Olist, a Brazilian online marketplace.
- **Problem:** customer review scores are falling and the COO suspects late deliveries.
- **Decision:** where to act (which states, sellers, categories) to cut late deliveries and raise review scores.
- **Success measure:** lower late-delivery rate and higher average review score.

| Q | Question |
|---|---|
| Q1 | How did monthly orders and revenue change over time (3-month rolling average)? |
| Q2 | Delivery days and days late per order; share of late orders; worst states? |
| Q3 | Do review scores differ between on-time and late orders (t-test)? |
| Q4 | Which categories bring the most revenue and which get the worst reviews? |
| Q5 | Among sellers with >= 30 orders, who has the highest late rate, and what share of late orders do the worst 10% cause? |

In [29]:
from pathlib import Path
hits = list(Path("/content/drive/MyDrive").rglob("olist_orders_dataset.csv"))
print(hits)

[PosixPath('/content/drive/MyDrive/MyDrive olist/olist_orders_dataset.csv')]


In [30]:
DATA = Path("/content/drive/MyDrive/MyDrive olist")

In [31]:
for f in sorted(DATA.iterdir()):
    print(f.name)

olist_customers_dataset.csv
olist_geolocation_dataset.csv
olist_order_items_dataset.csv
olist_order_payments_dataset.csv
olist_order_reviews_dataset.csv
olist_orders_dataset.csv
olist_products_dataset.csv
olist_sellers_dataset.csv
product_category_name_translation.csv


In [32]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy import stats
import plotly.express as px
import plotly.graph_objects as go

# Looks in the relative folders first (for markers), then your Google Drive folder
DATA = next((p for p in [Path("../data"), Path("data"), Path("."),
                         Path("/content/drive/MyDrive/MyDrive olist")]
             if (p / "olist_orders_dataset.csv").exists()), None)
assert DATA is not None, "Olist CSV files not found. On Colab, run drive.mount('/content/drive') first."
print("Using data folder:", DATA.resolve())

Using data folder: /content/drive/MyDrive/MyDrive olist


## 2. Data loading

In [33]:
orders    = pd.read_csv(DATA / "olist_orders_dataset.csv")
items     = pd.read_csv(DATA / "olist_order_items_dataset.csv")
reviews   = pd.read_csv(DATA / "olist_order_reviews_dataset.csv")
customers = pd.read_csv(DATA / "olist_customers_dataset.csv")
sellers   = pd.read_csv(DATA / "olist_sellers_dataset.csv")
products  = pd.read_csv(DATA / "olist_products_dataset.csv")
cat_tr    = pd.read_csv(DATA / "product_category_name_translation.csv")
geo       = pd.read_csv(DATA / "olist_geolocation_dataset.csv")   # used for the map

for name, df in [("orders", orders), ("items", items), ("reviews", reviews),
                 ("customers", customers), ("sellers", sellers), ("products", products)]:
    print(f"{name:10s} {df.shape}")

orders     (99441, 8)
items      (112650, 7)
reviews    (99224, 7)
customers  (99441, 5)
sellers    (3095, 4)
products   (32951, 9)


## 3. Cleaning
Every step is recorded in a cleaning log: step, reason, rows before and after.

In [34]:
cleaning_log = []

def log_step(step, reason, before, after):
    cleaning_log.append({"Step": step, "Reason": reason,
                         "Rows before": before, "Rows after": after,
                         "Rows removed": before - after})

# 1. Convert timestamps
date_cols = ["order_purchase_timestamp", "order_approved_at",
             "order_delivered_carrier_date", "order_delivered_customer_date",
             "order_estimated_delivery_date"]
orders[date_cols] = orders[date_cols].apply(pd.to_datetime)
log_step("Convert timestamps to datetime",
         "Needed for date maths (delivery days, monthly trend)", len(orders), len(orders))

# 2. Delivered orders only
before = len(orders)
orders = orders[orders["order_status"] == "delivered"].copy()
log_step("Keep order_status == 'delivered'",
         "Only delivered orders have a real delivery date to judge lateness", before, len(orders))

# 3. Missing delivery date
before = len(orders)
orders = orders.dropna(subset=["order_delivered_customer_date"])
log_step("Drop missing order_delivered_customer_date",
         "Cannot compute delivery days or lateness without it", before, len(orders))

# 4. New columns
orders["delivery_days"] = (orders["order_delivered_customer_date"] - orders["order_purchase_timestamp"]).dt.days
orders["days_late"] = (orders["order_delivered_customer_date"] - orders["order_estimated_delivery_date"]).dt.days
orders["is_late"] = orders["order_delivered_customer_date"] > orders["order_estimated_delivery_date"]
orders["order_month"] = orders["order_purchase_timestamp"].dt.to_period("M").dt.to_timestamp()
log_step("Create delivery_days, days_late, is_late, order_month",
         "Core variables (late = delivered after the estimated date)", len(orders), len(orders))

# 5. Outliers: 99th percentile is ~46 days, max is 209; cut at 120 to remove only extreme cases
before = len(orders)
print("99th percentile delivery days:", orders["delivery_days"].quantile(0.99))
orders = orders[(orders["delivery_days"] >= 0) & (orders["delivery_days"] <= 120)]
log_step("Remove delivery_days < 0 or > 120",
         "Extreme values (99th pct = 46 days, max = 209) distort averages; only a handful removed",
         before, len(orders))

# 6. One review per order
before = len(reviews)
reviews["review_answer_timestamp"] = pd.to_datetime(reviews["review_answer_timestamp"])
reviews = reviews.sort_values("review_answer_timestamp").drop_duplicates("order_id", keep="last")
log_step("Keep 1 review per order (latest)",
         "Some orders have several reviews; would duplicate rows on merge", before, len(reviews))

# 7. Merge customers and reviews (order level)
before = len(orders)
order_df = (orders.merge(customers[["customer_id", "customer_state"]], on="customer_id", how="left")
                  .merge(reviews[["order_id", "review_score"]], on="order_id", how="left"))
assert len(order_df) == before, "Merge changed row count!"
log_step("Merge customers (state) and reviews",
         "Need state and review score per order; row count checked unchanged", before, len(order_df))

# 8. Orders without review
before = len(order_df)
order_df = order_df.dropna(subset=["review_score"])
log_step("Drop orders without a review score",
         "Review-based questions (Q3, Q4) need a score", before, len(order_df))

99th percentile delivery days: 46.0


In [35]:
# Items-level table (several rows per order, so aggregate before counting orders/revenue)
items_df = items[items["order_id"].isin(order_df["order_id"])].copy()

products = products.merge(cat_tr, on="product_category_name", how="left")
n_missing_en = products["product_category_name_english"].isna().sum()
products["category"] = products["product_category_name_english"].fillna("unknown")
log_step("Fill missing English category with 'unknown'",
         f"{n_missing_en} products have no English name (translation table incomplete)",
         len(products), len(products))

before = len(items_df)
items_df = (items_df.merge(products[["product_id", "category"]], on="product_id", how="left")
                    .merge(sellers[["seller_id", "seller_state"]], on="seller_id", how="left")
                    .merge(order_df[["order_id", "order_month", "customer_state",
                                     "is_late", "review_score"]], on="order_id", how="left"))
assert len(items_df) == before
items_df["category"] = items_df["category"].fillna("unknown")

rev = items_df.groupby("order_id").agg(revenue=("price", "sum"),
                                       freight=("freight_value", "sum")).reset_index()
order_df = order_df.merge(rev, on="order_id", how="left")
assert order_df["order_id"].is_unique

log_df = pd.DataFrame(cleaning_log)
display(log_df)
print("Order-level table:", order_df.shape, "| items table:", items_df.shape)

,Step,Reason,Rows before,Rows after,Rows removed
0,Convert timestamps to datetime,"Needed for date maths (delivery days, monthly ...",99441,99441,0
1,Keep order_status == 'delivered',Only delivered orders have a real delivery dat...,99441,96478,2963
2,Drop missing order_delivered_customer_date,Cannot compute delivery days or lateness witho...,96478,96470,8
3,"Create delivery_days, days_late, is_late, orde...",Core variables (late = delivered after the est...,96470,96470,0
4,Remove delivery_days < 0 or > 120,"Extreme values (99th pct = 46 days, max = 209)...",96470,96427,43
5,Keep 1 review per order (latest),Some orders have several reviews; would duplic...,99224,98673,551
6,Merge customers (state) and reviews,Need state and review score per order; row cou...,96427,96427,0
7,Drop orders without a review score,"Review-based questions (Q3, Q4) need a score",96427,95783,644
8,Fill missing English category with 'unknown',623 products have no English name (translation...,32951,32951,0


Order-level table: (95783, 16) | items table: (109316, 13)


## 4. Analysis

### Q1: Monthly orders and revenue with 3-month rolling average

In [36]:
monthly = (order_df.groupby("order_month")
                   .agg(orders=("order_id", "nunique"), revenue=("revenue", "sum"))
                   .reset_index())
# Keep complete months only: 2016 is nearly empty and the data stops in Aug/Sep 2018
monthly = monthly[(monthly["order_month"] >= "2017-01-01") &
                  (monthly["order_month"] <= "2018-08-01")].copy()
monthly["orders_roll3"]  = monthly["orders"].rolling(3).mean()
monthly["revenue_roll3"] = monthly["revenue"].rolling(3).mean()
display(monthly)

,order_month,orders,revenue,orders_roll3,revenue_roll3
3,2017-01-01,741,110660.00,NaN,NaN
4,2017-02-01,1641,232905.22,NaN,NaN
5,2017-03-01,2516,351029.29,1632.666667,231531.503333
6,2017-04-01,2284,337849.37,2147.000000,307261.293333
7,2017-05-01,3514,483794.48,2771.333333,390891.046667
8,2017-06-01,3109,419059.15,2969.000000,413567.666667
9,2017-07-01,3842,477022.56,3488.333333,459958.730000
10,2017-08-01,4165,548476.37,3705.333333,481519.360000
11,2017-09-01,4117,603575.68,4041.333333,543024.870000
12,2017-10-01,4446,641918.45,4242.666667,597990.166667


### Q2: Delivery days, days late, late share, worst states (own function)

In [37]:
def calc_late_rate(df, group_col):
    """Late-order share (%) and order count per group, sorted worst first."""
    out = (df.groupby(group_col)
             .agg(orders=("order_id", "nunique"), late_rate=("is_late", "mean"))
             .reset_index())
    out["late_rate"] = (out["late_rate"] * 100).round(2)
    return out.sort_values("late_rate", ascending=False)

platform_late = order_df["is_late"].mean()
print(f"Late share overall: {platform_late:.2%}")
print(f"Delivery days: mean {order_df['delivery_days'].mean():.1f}, median {order_df['delivery_days'].median():.0f}")
state_late = calc_late_rate(order_df, "customer_state")
display(state_late.head(10))

Late share overall: 7.96%
Delivery days: mean 12.0, median 10


,customer_state,orders,late_rate
1,AL,394,23.35
9,MA,711,19.13
16,PI,469,15.57
5,CE,1271,15.11
24,SE,332,14.46
4,BA,3222,13.53
18,RJ,12206,13.26
26,TO,273,12.82
13,PA,932,11.80
7,ES,1966,11.75


### Q3: Review scores, on-time vs late (Welch t-test)

In [38]:
on_time = order_df.loc[~order_df["is_late"], "review_score"]
late    = order_df.loc[order_df["is_late"], "review_score"]
t_stat, p_value = stats.ttest_ind(on_time, late, equal_var=False)
diff = on_time.mean() - late.mean()
pooled_sd = np.sqrt((on_time.var(ddof=1) + late.var(ddof=1)) / 2)
cohens_d = diff / pooled_sd

p_text = "p < 0.001" if p_value < 0.001 else f"p = {p_value:.3f}"
print(f"On-time: n={len(on_time):,}, mean={on_time.mean():.2f} | Late: n={len(late):,}, mean={late.mean():.2f}")
print(f"t = {t_stat:.1f}, {p_text}, difference = {diff:.2f} stars, Cohen's d = {cohens_d:.2f}")
print(f"\nManager sentence: Late orders score about {diff:.1f} stars lower than on-time orders "
      f"({late.mean():.1f} vs {on_time.mean():.1f}), and a gap this big is extremely unlikely to be chance.")

On-time: n=88,163, mean=4.29 | Late: n=7,620, mean=2.56
t = 89.3, p < 0.001, difference = 1.73 stars, Cohen's d = 1.21

Manager sentence: Late orders score about 1.7 stars lower than on-time orders (2.6 vs 4.3), and a gap this big is extremely unlikely to be chance.


### Q4: Revenue and reviews by category

In [39]:
cat_orders = items_df.drop_duplicates(["order_id", "category"])   # 1 row per order-category
cat_stats = (items_df.groupby("category").agg(revenue=("price", "sum")).reset_index()
             .merge(cat_orders.groupby("category")
                              .agg(orders=("order_id", "nunique"),
                                   avg_review=("review_score", "mean"),
                                   late_rate=("is_late", "mean")).reset_index(),
                    on="category"))
cat_stats["late_rate"] = (cat_stats["late_rate"] * 100).round(1)

print("Top 10 by revenue:")
display(cat_stats.sort_values("revenue", ascending=False).head(10).round(2))
print("Worst 10 by average review (min 100 orders), with late rate as a possible cause:")
display(cat_stats[cat_stats["orders"] >= 100].sort_values("avg_review").head(10).round(2))

Top 10 by revenue:


,category,revenue,orders,avg_review,late_rate
43,health_beauty,1224603.68,8600,4.23,8.8
71,watches_gifts,1158251.65,5449,4.12,8.3
7,bed_bath_table,1013418.92,9174,4.00,8.6
65,sports_leisure,947971.55,7482,4.23,7.6
15,computers_accessories,884087.25,6496,4.08,7.6
39,furniture_decor,705678.74,6256,4.06,8.3
49,housewares,611560.80,5707,4.19,6.8
20,cool_stuff,603011.16,3529,4.22,6.6
5,auto,570670.35,3788,4.15,8.4
69,toys,466460.16,3773,4.24,7.3


Worst 10 by average review (min 100 orders), with late rate as a possible cause:


,category,revenue,orders,avg_review,late_rate
57,office_furniture,264910.08,1242,3.64,8.9
30,fashion_male_clothing,10257.33,105,3.82,3.8
4,audio,50290.91,345,3.84,12.8
47,home_confort,57141.65,389,3.89,10.3
34,fixed_telephony,54715.24,209,3.97,4.8
19,construction_tools_safety,38744.22,158,3.97,4.4
48,home_construction,80756.07,480,3.99,7.9
7,bed_bath_table,1013418.92,9174,4.00,8.6
70,unknown,173035.94,1400,4.01,8.8
33,fashion_underwear_beach,9256.05,116,4.01,12.9


### Q5: Seller late rates (>= 30 orders) and the worst 10%

In [40]:
seller_orders = items_df.drop_duplicates(["order_id", "seller_id"])   # 1 row per order-seller
seller_stats = (seller_orders.groupby("seller_id")
                .agg(orders=("order_id", "nunique"), late_orders=("is_late", "sum"),
                     late_rate=("is_late", "mean"), avg_review=("review_score", "mean")).reset_index())
qualified = seller_stats[seller_stats["orders"] >= 30].copy()

cutoff = qualified["late_rate"].quantile(0.90)
worst = qualified[qualified["late_rate"] >= cutoff].copy()
total_late_all = seller_stats["late_orders"].sum()
share_all = worst["late_orders"].sum() / total_late_all
share_qual = worst["late_orders"].sum() / qualified["late_orders"].sum()

print(f"Sellers: {len(seller_stats):,} total, {len(qualified):,} with >= 30 orders")
print(f"Worst 10% = {len(worst)} sellers (late rate >= {cutoff:.1%})")
print(f"They cause {share_all:.1%} of ALL late orders ({share_qual:.1%} of late orders among qualified sellers)")

# Impact if the worst sellers were only as late as the platform average
seller_platform_rate = seller_orders["is_late"].mean()
worst["excess_late"] = worst["late_orders"] - worst["orders"] * seller_platform_rate
avoided_sellers = worst["excess_late"].sum()
print(f"If they matched the platform late rate ({seller_platform_rate:.1%}), "
      f"late orders would fall by about {avoided_sellers:,.0f} ({avoided_sellers/total_late_all:.1%} of all late orders)")

Sellers: 2,964 total, 623 with >= 30 orders
Worst 10% = 63 sellers (late rate >= 14.4%)
They cause 12.0% of ALL late orders (14.5% of late orders among qualified sellers)
If they matched the platform late rate (7.9%), late orders would fall by about 529 (6.9% of all late orders)


## 5. Visualisation
Ten charts, seven types (line, histogram, bar, map, box, stacked bar, scatter). Charts 1, 5 and 9 are interactive Plotly charts (all are Plotly, so all are interactive).
Replace each **Takeaway** with one sentence based on your own output.

**Chart 1: monthly orders (Q1)**

In [41]:
fig = px.line(monthly, x="order_month", y=["orders", "orders_roll3"],
              title="Monthly orders with 3-month rolling average",
              labels={"order_month": "Month", "value": "Orders per month", "variable": ""})
fig.show()

**Takeaway:** TODO (e.g. orders grew from about 740 in Jan 2017 to about 6,000-7,000 a month by 2018, then flattened).

**Chart 2: monthly revenue (Q1)**

In [42]:
fig = px.line(monthly, x="order_month", y=["revenue", "revenue_roll3"],
              title="Monthly revenue with 3-month rolling average",
              labels={"order_month": "Month", "value": "Revenue (BRL)", "variable": ""})
fig.show()

**Takeaway:** TODO (revenue follows orders: about 111k BRL in Jan 2017 to about 850k-970k in 2018).

**Chart 3: delivery time distribution (Q2)**

In [43]:
fig = px.histogram(order_df[order_df["delivery_days"] <= 60], x="delivery_days", nbins=60,
                   title="Distribution of delivery time (orders up to 60 days)",
                   labels={"delivery_days": "Delivery time (days)"})
fig.update_yaxes(title="Number of orders")
fig.add_vline(x=order_df["delivery_days"].median(), line_dash="dash",
              annotation_text="median")
fig.show()

**Takeaway:** TODO (median delivery is 10 days, mean 12; a long right tail of slow orders).

**Chart 4: late rate by state (Q2)**

In [44]:
fig = px.bar(state_late, x="customer_state", y="late_rate", hover_data=["orders"],
             title="Late-delivery rate by customer state",
             labels={"customer_state": "State", "late_rate": "Late orders (%)"})
fig.add_hline(y=platform_late * 100, line_dash="dash", annotation_text="platform average")
fig.show()

**Takeaway:** TODO (name the 3 worst states and their late rate vs the 8% average).

**Chart 5: late-rate map (bonus, Q2)**

In [45]:
state_pos = (geo.groupby("geolocation_state")[["geolocation_lat", "geolocation_lng"]]
                .median().reset_index().rename(columns={"geolocation_state": "customer_state"}))
map_df = state_late.merge(state_pos, on="customer_state")
fig = px.scatter_geo(map_df, lat="geolocation_lat", lon="geolocation_lng",
                     size="orders", color="late_rate", hover_name="customer_state",
                     color_continuous_scale="Reds", scope="south america",
                     title="Late-delivery rate by state (bubble size = number of orders)",
                     labels={"late_rate": "Late orders (%)"})
fig.update_geos(fitbounds="locations")
fig.show()

**Takeaway:** TODO (e.g. the north-east has the highest late rates; the big south-east states are near average).

**Chart 6: delivery days by review score (Q3)**

In [46]:
order_df["score_label"] = order_df["review_score"].astype(int).astype(str)
fig = px.box(order_df, x="score_label", y="delivery_days", points=False,
             category_orders={"score_label": ["1", "2", "3", "4", "5"]},
             title="Delivery time by review score",
             labels={"score_label": "Review score (stars)", "delivery_days": "Delivery time (days)"})
fig.update_yaxes(range=[0, 50])
fig.show()

**Takeaway:** TODO (1-star orders take clearly longer to arrive than 5-star orders).

**Chart 7: review score mix, on-time vs late (Q3)**

In [47]:
order_df["status"] = np.where(order_df["is_late"], "Late", "On time")
share = (pd.crosstab(order_df["status"], order_df["review_score"], normalize="index") * 100).round(1)
share = share.reset_index().melt(id_vars="status", var_name="review_score", value_name="share")
share["review_score"] = share["review_score"].astype(int).astype(str)
fig = px.bar(share, x="status", y="share", color="review_score", barmode="stack",
             title="Share of review scores: on-time vs late orders",
             labels={"status": "Delivery status", "share": "Share of orders (%)", "review_score": "Stars"})
fig.show()

**Takeaway:** TODO (late orders get mostly 1-2 stars; on-time orders get mostly 5 stars. Mean 2.56 vs 4.29).

**Chart 8: top categories by revenue (Q4)**

In [48]:
top_cat = cat_stats.sort_values("revenue", ascending=False).head(10).sort_values("revenue")
fig = px.bar(top_cat, x="revenue", y="category", orientation="h",
             title="Top 10 product categories by revenue",
             labels={"revenue": "Revenue (BRL)", "category": "Category"})
fig.show()

**Takeaway:** TODO (health_beauty, watches_gifts and bed_bath_table lead revenue at about 1.0-1.2M BRL each).

**Chart 9: revenue vs review score by category (Q4)**

In [49]:
fig = px.scatter(cat_stats[cat_stats["orders"] >= 100], x="revenue", y="avg_review",
                 size="orders", color="late_rate", hover_name="category", log_x=True,
                 color_continuous_scale="Reds",
                 title="Revenue vs average review score by category (colour = late rate)",
                 labels={"revenue": "Revenue (BRL, log scale)", "avg_review": "Average review (stars)",
                         "late_rate": "Late orders (%)"})
fig.show()

**Takeaway:** TODO (office_furniture has the worst reviews at 3.64; say whether its late rate explains it).

**Chart 10: worst sellers by late rate (Q5)**

In [50]:
top_sellers = qualified.sort_values("late_rate", ascending=False).head(15).copy()
top_sellers["seller"] = top_sellers["seller_id"].str[:8]
top_sellers = top_sellers.sort_values("late_rate")
top_sellers["late_rate_pct"] = top_sellers["late_rate"] * 100
fig = px.bar(top_sellers, x="late_rate_pct", y="seller", orientation="h",
             color="avg_review", color_continuous_scale="RdYlGn", hover_data=["orders"],
             title="15 sellers with the highest late rate (sellers with 30+ orders)",
             labels={"late_rate_pct": "Late orders (%)", "seller": "Seller (first 8 chars of ID)",
                     "avg_review": "Average review"})
fig.show()

**Takeaway:** TODO (late rates of 23-33% at the top; the worst 10% of sellers cause 12.0% of all late orders, so lateness is widespread).

## 6. Recommendations and impact

In [51]:
n = len(order_df)
star_gap = diff                                   # stars lost when an order is late
avg_review_now = order_df["review_score"].mean()

# R1: halve the platform late rate
saved1 = order_df["is_late"].sum() * 0.5
gain1 = saved1 * star_gap / n
print(f"R1: halving late deliveries saves about {saved1:,.0f} late orders -> +{gain1:.2f} stars "
      f"(from {avg_review_now:.2f}); removing ALL lateness would give +{order_df['is_late'].sum()*star_gap/n:.2f}")

# R2: fix the worst 10% of sellers
gain2 = avoided_sellers * star_gap / n
print(f"R2: worst-seller fix avoids about {avoided_sellers:,.0f} late orders -> +{gain2:.3f} stars")

# R3: raise the worst-reviewed big category to the platform average
worst_cat = cat_stats[cat_stats["orders"] >= 100].sort_values("avg_review").iloc[0]
gain3 = worst_cat["orders"] * (avg_review_now - worst_cat["avg_review"]) / n
print(f"R3: {worst_cat['category']} ({worst_cat['orders']:,.0f} orders, {worst_cat['avg_review']:.2f} stars, "
      f"{worst_cat['late_rate']:.1f}% late) -> platform-average reviews would add +{gain3:.3f} stars overall")

R1: halving late deliveries saves about 3,810 late orders -> +0.07 stars (from 4.16); removing ALL lateness would give +0.14
R2: worst-seller fix avoids about 529 late orders -> +0.010 stars
R3: office_furniture (1,242 orders, 3.64 stars, 8.9% late) -> platform-average reviews would add +0.007 stars overall


Write your 3+ recommendations here, each with the number and estimated impact printed above, for example:

1. **Cut late deliveries platform-wide (states with the highest late rate first).** Late orders are 8.0% of orders and score 1.7 stars lower; halving them is worth about +X stars.
2. **Put the worst 10% of sellers on an improvement plan.** They cause 12% of late orders; matching the average would avoid about Y late orders.
3. **Investigate office_furniture (and bed_bath_table, a top-3 revenue category at 4.00).** Worst reviews among large categories.

**Limitations**
- The t-test shows association, not proof that lateness alone causes low scores.
- Impact figures assume the 1.7-star gap holds when lateness is removed (an upper bound).
- Only delivered orders with a review are analysed (about 95.8k of 99.4k orders); 2016 and late 2018 are excluded from the trend.
- "Late" uses Olist's own estimated date, which Olist can change.

## 7. Save cleaned data for the Streamlit app

In [52]:
order_df.to_csv(DATA / "orders_clean.csv.gz", index=False, compression="gzip")
seller_orders.to_csv(DATA / "seller_orders_clean.csv.gz", index=False, compression="gzip")
print("Saved cleaned files to", DATA.resolve())

Saved cleaned files to /content/drive/MyDrive/MyDrive olist


In [53]:
# Main category per order = category of its most expensive item (needed for the app's category filter)
main_cat = (items_df.sort_values("price", ascending=False)
                    .drop_duplicates("order_id")[["order_id", "category"]])
orders_save = order_df.merge(main_cat, on="order_id", how="left")[
    ["order_id", "order_purchase_timestamp", "order_month", "customer_state", "category",
     "review_score", "revenue", "freight", "is_late", "delivery_days", "days_late"]]

# One row per (order, seller) with the seller's revenue on that order
seller_save = (items_df.groupby(["order_id", "seller_id"]).agg(revenue=("price", "sum")).reset_index()
                       .merge(order_df[["order_id", "order_month", "is_late", "review_score"]], on="order_id"))

orders_save.to_csv(DATA / "orders_clean.csv.gz", index=False, compression="gzip")
seller_save.to_csv(DATA / "seller_orders_clean.csv.gz", index=False, compression="gzip")
print("Saved cleaned files to", DATA.resolve(), "|", orders_save.shape, seller_save.shape)

Saved cleaned files to /content/drive/MyDrive/MyDrive olist | (95783, 11) (97110, 6)
